# 1. Beispieldaten erzeugen

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [8]:
df = pd.DataFrame({
    "aktie_a": [0.02, 0.01, -0.01, 0.03],
    "aktie_b": [0.01, -0.02, 0.02, 0.04],
    "aktie_c": [0.05, -0.02, 0.02, 0.04]
})
print(type(df))
print(type(df["aktie_a"]))

<class 'pandas.DataFrame'>
<class 'pandas.Series'>


# 2. "pandas.Series" UND "pandas.Dataframe" INS "numpy.ndarray"

In [ ]:
werte = df["aktie_a"].to_numpy()
renditen = df[["aktie_a", "aktie_b", "aktie_c"]].to_numpy()
print("Objektyp",type(werte))
print("Objektyp",type(renditen))
print("1-D-Array:\n", werte)
print("----")
print("2-D-Array:\n", renditen)

Objektyp <class 'numpy.ndarray'>
Objektyp <class 'numpy.ndarray'>
1-D-Array:
 [ 0.02  0.01 -0.01  0.03]
----
2-D-Array:
 [[ 0.02  0.01  0.05]
 [ 0.01 -0.02 -0.02]
 [-0.01  0.02  0.02]
 [ 0.03  0.04  0.04]]


# 3. NumPy-Kovarianzmatrix aus einem pandas.DataFrame und/oder numpy.2array

- np.cov() betrachtet standardmäßig jede Zeile als Variable. Bei deinem DataFrame (bzw. in daraus erzeugte np matrix (numpy.2array)s. oben) befinden sich die Variablen in den Spalten.

In [ ]:
kovarianzmatrix = np.cov(renditen, rowvar=False, ddof=1)
# ddof=1 bedeutet: Wir verwenden die Stichprobenkovarianz, nicht die Populationskovarianz.
print(kovarianzmatrix)
# rowvar=False bedeutet: Die Variablen beziehungsweise Aktien stehen in den Spalten. 

[[0.00029167 0.000125   0.000225  ]
 [0.000125   0.000625   0.00059167]
 [0.000225   0.00059167 0.00095833]]


- Einfacher mit pandas: Für einen normalen DataFrame ist häufig diese Variante verständlicher:

In [10]:
kovarianzmatrix = df[
    ["aktie_a", "aktie_b", "aktie_c"]
].cov()
#Vorteil: Das Ergebnis ist wieder ein DataFrame mit Beschriftungen:
# Für Matrixrechnungen und Optimierung: kovarianz_array = kovarianz_df.to_numpy()

# 4. NumPy-Ergebnis wieder in pandas umwandeln

## A. Eindimensionales Array zu Series

In [11]:
array = np.array([0.02, 0.01, -0.01, 0.03])

serie = pd.Series(array, name="aktie_a")

In [12]:
serie = pd.Series(
    array,
    index=df.index,
    name="aktie_a"
)

## B. Zweidimensionales Array zu DataFrame

In [14]:
array = np.array([
[0.02, 0.01],
[0.01, -0.02],
[-0.01, 0.02]
])

neuer_df = pd.DataFrame(
array,
columns=["aktie_a", "aktie_b"]
)
print(neuer_df)

   aktie_a  aktie_b
0     0.02     0.01
1     0.01    -0.02
2    -0.01     0.02


In [ ]:
neuer_df = pd.DataFrame(
    array,
    index=df.index[:3],
    # index aus dem Original-DataFrame übernehmen
    columns=["aktie_a", "aktie_b"]
)
print(neuer_df)

   aktie_a  aktie_b
0     0.02     0.01
1     0.01    -0.02
2    -0.01     0.02


In [21]:
spalten = ["aktie_a", "aktie_b", "aktie_c"]

kovarianz_array = np.cov(
    df[spalten].to_numpy(),
    rowvar=False,
    ddof=1
)

kovarianz_df = pd.DataFrame(
    kovarianz_array,
    index=spalten,
    columns=spalten
)

# 5. Probleme und Herausforderungen

## A. Fehlende Werte

- Angenommen, eine Spalte enthält NaN:

```Python
df = pd.DataFrame({
"aktie_a": [0.02, 0.01, np.nan, 0.03],
"aktie_b": [0.01, np.nan, 0.02, 0.04]
})
```
-  Diese SciPy-Berechnung kann dann ein ungültiges Ergebnis liefern:

Python
stats.pearsonr(df["aktie_a"], df["aktie_b"])


- Falsch: Spalten getrennt bereinigen
```Python
a = df["aktie_a"].dropna()
b = df["aktie_b"].dropna()
```
- Das kann problematisch sein, weil danach Renditen aus unterschiedlichen Zeitpunkten miteinander verglichen werden.

In [17]:
# Richtig: gemeinsame Zeilen auswählen
daten = df[["aktie_a", "aktie_b"]].dropna()

korrelation, p_wert = stats.pearsonr(
    daten["aktie_a"].to_numpy(),
    daten["aktie_b"].to_numpy()
)

- Alternative mit nan_policy

-   Einige SciPy-Funktionen unterstützen:

    ```Python
    stats.ttest_ind(
    df["aktie_a"],
    df["aktie_b"],
    nan_policy="omit"
    )
    ```

- Aber nicht jede SciPy-Funktion besitzt denselben Umgang mit fehlenden Werten. Gemeinsames dropna() ist bei zusammengehörigen Finanzzeitreihen meist transparenter.

## B. Unterschiedliche Längen

- Das geht nicht:

    ```Python
    a = np.array([1, 2, 3])
    b = np.array([1, 2])
    stats.pearsonr(a, b)
    ```
- Beide Variablen müssen gleich viele zusammengehörige Beobachtungen haben.

## C. Zweidimensional statt eindimensional

- Das hier ist ein DataFrame:

```Python
df[["aktie_a"]]
```
- Es besitzt die Form:

```Plain Text
(4, 1)
```
- Das hier ist eine Series:

```Python
df["aktie_a"]
```
- Sie besitzt die Form:

```Plain Text
(4,)
```
- Viele statistische Funktionen erwarten für eine einzelne Variable (4,), also eindimensionale Daten.

- Umwandlung:

```Python
werte = df[["aktie_a"]].to_numpy().ravel()
```
- Einfacher:

```Python
werte = df["aktie_a"].to_numpy()
```

## D. Indexverlust

- Nach:

In [21]:
array = df["aktie_a"].to_numpy()
print("Array:", array)


Array: [ 0.02  0.01 -0.01  0.03]


  - kennt das Array keine Datumsangaben und keine Zeilennamen mehr.

-  Wenn du das Ergebnis zurückschreibst:

In [22]:
df["neues_ergebnis"] = array
print(df)

   aktie_a  aktie_b  aktie_c  neues_ergebnis
0     0.02     0.01     0.05            0.02
1     0.01    -0.02    -0.02            0.01
2    -0.01     0.02     0.02           -0.01
3     0.03     0.04     0.04            0.03


- ordnet pandas es nur anhand der Position zu.

- Sicherer bei einem separat berechneten Ergebnis:

In [23]:
ergebnis = pd.Series(
array,
index=df.index,
name="neues_ergebnis"
)
print(ergebnis)

0    0.02
1    0.01
2   -0.01
3    0.03
Name: neues_ergebnis, dtype: float64
